In [3]:
import os, pickle, signal, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from tqdm import tqdm
import safe as sf

/data/ryanschen/safe-retro/saferetrouv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
BASE = "/data/ryanschen/safe-retro"
ONMT_TRANSLATE = f"{BASE}/saferetrouv/bin/onmt_translate"
TEST_CSV = "/data/ryanschen/safe-retro/Data/test_safe.csv"

MODELS = {
    "SMILES":   f"{BASE}/smiles_run/model_step_400000.pt",
    "SAFE+Aug": f"{BASE}/safe_aug_run/model_step_400000.pt",
}
SRC_TEST = {
    "SMILES":   f"{BASE}/USPTO_SMILES_preprocessed/src-test.txt",
    "SAFE+Aug": f"{BASE}/USPTO_SAFE_preprocessed/src-test.txt",
}
PRED_FILES = {
    "SMILES":   f"{BASE}/smiles_run/predictions.txt",
    "SAFE+Aug": f"{BASE}/safe_aug_run/predictions.txt",
}
CACHE_FILES = {
    "SMILES":   f"{BASE}/smiles_run/eval_cache.pkl",
    "SAFE+Aug": f"{BASE}/safe_aug_run/eval_cache.pkl",
}

SAFE_MODELS    = {"SAFE", "SAFE+Aug"}
N_BEST         = 10
BEAM_SIZE      = 10
GPU            = 0       # set to -1 for CPU
DECODE_TIMEOUT = 5       # seconds per safe.decode call
COLORS = {"SMILES": "#4C72B0", "SAFE": "#DD8452", "SAFE+Aug": "#55A868"}

test_df = pd.read_csv(TEST_CSV)
print(f"Test set: {len(test_df):,} reactions | columns: {list(test_df.columns)}")

Test set: 39,994 reactions | columns: ['precursors', 'products', 'split', 'rxn_smiles', 'safe']


In [5]:
# Check Files Exist
all_ok = True
for name in MODELS:
    for label, path in [("model", MODELS[name]), ("src-test", SRC_TEST[name])]:
        ok = os.path.exists(path)
        print(f"  {'OKKKK' if ok else 'MISSING'} [{name}] {label}: {path}")
        all_ok = all_ok and ok

ok_csv = os.path.exists(TEST_CSV)
print(f"  {'OKKKK' if ok_csv else 'MISSING'} test.csv: {TEST_CSV}")

if all_ok and ok_csv:
    print("Can now run inference")
else:
    print("Stuff is missing")

  OKKKK [SMILES] model: /data/ryanschen/safe-retro/smiles_run/model_step_400000.pt
  OKKKK [SMILES] src-test: /data/ryanschen/safe-retro/USPTO_SMILES_preprocessed/src-test.txt
  OKKKK [SAFE+Aug] model: /data/ryanschen/safe-retro/safe_aug_run/model_step_400000.pt
  OKKKK [SAFE+Aug] src-test: /data/ryanschen/safe-retro/USPTO_SAFE_preprocessed/src-test.txt
  OKKKK test.csv: /data/ryanschen/safe-retro/Data/test_safe.csv
Can now run inference


# Results of the Model (SAFE Augmented and SMILES Baseline)

In [26]:
# Decoding Helpers

def _timeout_handler(signum, frame):
    raise TimeoutError()

def to_canonical_smiles(smi: str) -> str:
    try:
        smi = smi.strip().replace(" ", "")
        # canonicalise each fragment separately then sort
        parts = smi.split(".")
        canon_parts = []
        for p in parts:
            mol = Chem.MolFromSmiles(p)
            if mol:
                canon_parts.append(Chem.MolToSmiles(mol))
        return ".".join(sorted(canon_parts)) if canon_parts else ""
    except Exception:
        return ""

def safe_to_canonical(safe_str: str) -> str:
    try:
        safe_str = safe_str.strip().replace(" ", "")
        # split on ~ (inter-molecule separator) and decode each molecule
        molecules = safe_str.split("~")
        canon_parts = []
        for mol_safe in molecules:
            signal.signal(signal.SIGALRM, _timeout_handler)
            signal.alarm(DECODE_TIMEOUT)
            try:
                mol = sf.decode(mol_safe, as_mol=True, ignore_errors=True)
                signal.alarm(0)
                if mol is not None:
                    canon_parts.append(Chem.MolToSmiles(mol))
            except Exception:
                signal.alarm(0)
                continue
        return ".".join(sorted(canon_parts)) if canon_parts else ""
    except Exception:
        return ""

print("Decoding helpers are done")

Decoding helpers are done


In [41]:
# Ground truth files — same format as model output
GT_FILES = {
    "SMILES": f"{BASE}/USPTO_SMILES_preprocessed/tgt-test.txt",
    "SAFE+Aug": f"{BASE}/USPTO_SAFE_preprocessed/tgt-test.txt",
}

def decode_predictions(name):
    cache = CACHE_FILES[name]
    if os.path.exists(cache):
        print(f"[{name}] Loading from cache...")
        with open(cache, "rb") as f:
            return pickle.load(f)

    decoder = safe_to_canonical if name in SAFE_MODELS else to_canonical_smiles

    # load predictions
    with open(PRED_FILES[name]) as f:
        raw = [line.strip() for line in f]

    # load ground truth from tgt-test.txt
    with open(GT_FILES[name]) as f:
        gt_raw = [line.strip() for line in f]

    n_actual = len(raw) // N_BEST
    n_gt = len(gt_raw)

    if len(raw) != n_gt * N_BEST:
        print(f"Warning: predictions {len(raw)} lines, GT {n_gt} lines")
        n_actual = min(n_actual, n_gt)

    print(f"[{name}] Evaluating {n_actual:,} reactions")

    grouped = [raw[i * N_BEST:(i+1) * N_BEST] for i in range(n_actual)]

    print(f"[{name}] Decoding predictions...")
    pred_smiles = [
        [decoder(p) for p in preds]
        for preds in tqdm(grouped, desc=name)
    ]

    print(f"[{name}] Decoding ground truth...")
    gt_smiles = [
        decoder(g)
        for g in tqdm(gt_raw[:n_actual], desc="GT")
    ]

    data = {"pred_smiles": pred_smiles, "gt_smiles": gt_smiles, "n": n_actual}
    with open(cache, "wb") as f:
        pickle.dump(data, f)
    print(f"[{name}] Cached to {cache}")
    return data

# delete old caches
for name in MODELS:
    if os.path.exists(CACHE_FILES[name]):
        os.remove(CACHE_FILES[name])
        print(f"Deleted cache: {CACHE_FILES[name]}")

# reload test_df just in case
test_df = pd.read_csv(TEST_CSV)
print(f"\nTest set: {len(test_df):,} reactions")

decoded = {}
for name in MODELS:
    decoded[name] = decode_predictions(name)
    print()


Test set: 39,994 reactions
[SMILES] Evaluating 39,994 reactions
[SMILES] Decoding predictions...


SMILES:   0%|          | 0/39994 [00:00<?, ?it/s][14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] SMILES Parse Error: unclosed ring for input: 'NC1CCN(CC2Cn3c(=O)ccc4ncc(F)c2c32)CC1O'
[14:14:20] SMILES Parse Error: unclosed ring for input: 'NC1CCN(CC2Cn3c(=O)ccc4ncc(F)c23)CC1O'
[14:14:20] SMILES Parse Error: unclosed ring for input: 'NC1CCN(CC2Cn3c(=O)ccc4ncc(F)c23)CC1O'
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WARNING: not removing hydrogen atom without neighbors
[14:14:20] WAR

[SMILES] Decoding ground truth...


GT:   0%|          | 0/39994 [00:00<?, ?it/s][14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen atom without neighbors
[14:15:19] WARNING: not removing hydrogen ato

[SMILES] Cached to /data/ryanschen/safe-retro/smiles_run/eval_cache.pkl

[SAFE+Aug] Evaluating 39,994 reactions
[SAFE+Aug] Decoding predictions...


SAFE+Aug: 100%|██████████| 39994/39994 [05:37<00:00, 118.65it/s]


[SAFE+Aug] Decoding ground truth...


GT: 100%|██████████| 39994/39994 [00:29<00:00, 1336.92it/s]


[SAFE+Aug] Cached to /data/ryanschen/safe-retro/safe_aug_run/eval_cache.pkl



In [42]:
# Taking off Reagents and Calculating Top-k Accuracy

def topk_accuracy(pred_list, gt_list, k):
    correct = sum(gt in preds[:k] for gt, preds in zip(gt_list, pred_list))
    return correct / len(gt_list)

def coverage(pred_list):
    return sum(any(p for p in preds) for preds in pred_list) / len(pred_list)

def valid_rate(pred_list):
    total = sum(len(p) for p in pred_list)
    valid = sum(1 for preds in pred_list for p in preds if p)
    return valid / total

_REAGENTS_EXTENDED = {
    # solvents
    "C1CCOC1", "CCO", "CO", "CCOC", "CC(C)O", "CC(O)=O", "CC#N",
    "ClCCl", "ClC(Cl)Cl", "c1ccccc1", "Cc1ccccc1", "CCOCC", "O",
    "CN(C)C=O", "CS(C)=O", "C1CCNCC1", "c1ccncc1", "Cc1ccccn1",
    # bases / acids
    "[Na+].[OH-]", "[K+].[OH-]", "CC(C)(C)[O-].[Na+]",
    "[O-]C(=O)[O-].[Na+].[Na+]", "O=C([O-])[O-].[K+].[K+]",
    # simple salts / atoms
    "[H-]", "[Na+]", "[K+]", "[Li+]", "Cl", "Br", "[F-]",
    "[NH4+].[Cl-]", "[H][H]", "O=O", "N",
    # common acids
    "O=S(=O)(O)O", "Cl.O", "OO",
}

_METALS = {"Li","Na","K","Cs","Mg","Ca","Al","Zn","Fe","Cu",
           "Pd","Ni","Rh","Ir","Ru","Os","Pt","Au","Ag","B"}

def strip_reagents(smi: str) -> str:
    if not smi:
        return ""
    parts = smi.split(".")
    core = []
    for p in parts:
        if p in _REAGENTS_EXTENDED:
            continue
        mol = Chem.MolFromSmiles(p)
        if mol is None:
            continue
        atoms = {a.GetSymbol() for a in mol.GetAtoms()}
        # skip pure metal/inorganic fragments
        if atoms & _METALS and not (atoms - _METALS - {"C","H","O","N","Cl","F","Br"}):
            continue
        # skip very small fragments (1-2 heavy atoms) — likely salts/counterions
        if mol.GetNumHeavyAtoms() <= 2:
            continue
        core.append(Chem.MolToSmiles(mol))
    return ".".join(sorted(core)) if core else smi

In [43]:
# Computation of Metrics

results = {}
for name in MODELS:
    pred = decoded[name]["pred_smiles"]
    gt   = decoded[name]["gt_smiles"]
    pred_core = [[strip_reagents(p) for p in preds] for preds in pred]
    gt_core   = [strip_reagents(g) for g in gt]
    results[name] = {
        "exact":      {k: topk_accuracy(pred, gt, k) for k in [1,3,5,10]},
        "core":       {k: topk_accuracy(pred_core, gt_core, k) for k in [1,3,5,10]},
        "coverage":   coverage(pred),
        "valid_rate": valid_rate(pred),
        "pred":       pred,
        "gt":         gt,
        "pred_core":  pred_core,
        "gt_core":    gt_core,
    }
print("Done.")

[14:21:52] WARNING: not removing hydrogen atom without neighbors
[14:21:53] WARNING: not removing hydrogen atom without neighbors
[14:21:53] WARNING: not removing hydrogen atom without neighbors
[14:22:03] WARNING: not removing hydrogen atom without neighbors
[14:22:04] WARNING: not removing hydrogen atom without neighbors
[14:22:04] WARNING: not removing hydrogen atom without neighbors
[14:22:04] WARNING: not removing hydrogen atom without neighbors
[14:22:04] WARNING: not removing hydrogen atom without neighbors
[14:22:06] WARNING: not removing hydrogen atom without neighbors
[14:22:06] WARNING: not removing hydrogen atom without neighbors
[14:22:09] WARNING: not removing hydrogen atom without neighbors
[14:22:10] WARNING: not removing hydrogen atom without neighbors
[14:22:10] WARNING: not removing hydrogen atom without neighbors
[14:22:10] WARNING: not removing hydrogen atom without neighbors
[14:22:10] WARNING: not removing hydrogen atom without neighbors
[14:22:16] WARNING: not r

Done.


In [44]:
# Print Results

names = list(MODELS.keys())

print(f"\n{'Metric':<26}", end="")
for name in names:
    print(f"{name:>13}", end="")
print()
print("-" * 52)

for k in [1, 3, 5, 10]:
    print(f"{'Top-'+str(k)+' exact match':<26}", end="")
    for name in names:
        print(f"{results[name]['exact'][k]*100:>12.2f}%", end="")
    print()

print()
for k in [1, 3, 5, 10]:
    print(f"{'Top-'+str(k)+' core-only':<26}", end="")
    for name in names:
        print(f"{results[name]['core'][k]*100:>12.2f}%", end="")
    print()

print()
print(f"{'Coverage':<26}", end="")
for name in names:
    print(f"{results[name]['coverage']*100:>12.2f}%", end="")
print()

print(f"{'Valid prediction rate':<26}", end="")
for name in names:
    print(f"{results[name]['valid_rate']*100:>12.2f}%", end="")
print()


Metric                           SMILES     SAFE+Aug
----------------------------------------------------
Top-1 exact match                18.31%        0.00%
Top-3 exact match                26.73%        0.00%
Top-5 exact match                30.09%        0.00%
Top-10 exact match               33.43%        0.00%

Top-1 core-only                  32.15%        0.01%
Top-3 core-only                  43.02%        0.02%
Top-5 core-only                  47.05%        0.02%
Top-10 core-only                 51.01%        0.02%

Coverage                        100.00%       99.99%
Valid prediction rate            99.99%       98.57%


In [27]:
import os
os.remove(CACHE_FILES["SAFE+Aug"])
print("Cache deleted")

Cache deleted


In [ ]:
# Tests
#Decoded safe aug against ground truth
for i in range(5):
    gt = decoded["SAFE+Aug"]["gt_smiles"][i]
    pred = decoded["SAFE+Aug"]["pred_smiles"][i][0]
    print(f"GT  : {gt}")
    print(f"P1  : {pred}")
    print(f"Match: {gt == pred}")
    print()

#

GT  : C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
P1  : Nc1cc(F)ccc1O.O=C(Cl)c1cc([N+](=O)[O-])ccc1F
Match: False

GT  : CCCCP(CCCC)CCCC.COC(=O)Cc1cn(C)c2cc(O)ccc12.Cc1ccccc1.Cc1nn(-c2ccc(C(F)(F)F)cc2)cc1C(C)CO
P1  : C1COCCO1.CC(C)c1nn(Cc2ccc(Br)cc2F)c(=O)c(C(=O)NCC(=O)O)c1O.Cl.O.O=C([O-])[O-].OB(O)c1ccc(C(F)(F)F)cc1.[K+].[K+].c1ccc([PH](c2ccccc2)(c2ccccc2)[Pd]([PH](c2ccccc2)(c2ccccc2)c2ccccc2)([PH](c2ccccc2)(c2ccccc2)c2ccccc2)[PH](c2ccccc2)(c2ccccc2)c2ccccc2)cc1
Match: False

GT  : CCOCC.Cl.Cl.ClC(Cl)Cl.NC1CCN(CC2Cn3c(=O)ccc4ncc(F)c2c43)CC1O.O=Cc1cc2c(cn1)OCS2
P1  : CCN(CC)CC
Match: False

GT  : C=C(C)C(=O)Cl.CC(C)=C1C(=O)N(c2ccc(O)cc2)C(=O)C1=C(C)c1cc(-c2ccccc2)sc1C.CCN(CC)CC.ClCCl
P1  : CCN(CC)CC.CCN=C=NCCCN(C)C.CN(C)C=O.Cl.Nc1nnn[nH]1.O.O.O=C(O)c1ccc(CN(c2ccc(Cl)cc2)c2nc(-c3ccc(Cl)cc3)cs2)cc1.On1nnc2ccccc21
Match: False

GT  : O=Cc1cncc(Cl)c1COC1CCCCO1.OCc1c(Cl)cncc1Cl
P1  : CN(C)C=O.O=P(Cl)(Cl)Cl
Match: False



In [38]:
# compare first line of both test src files
with open("/data/ryanschen/safe-retro/USPTO_SMILES_preprocessed/src-test.txt") as f:
    smiles_src = f.readline().strip()

with open("/data/ryanschen/safe-retro/USPTO_SAFE_preprocessed/src-test.txt") as f:
    safe_src = f.readline().strip()

print("SMILES src:", smiles_src)
print("SAFE src  :", safe_src)

print("GT precursors:", test_df["precursors"].iloc[0])
print("GT safe col  :", test_df["safe"].iloc[0])

SMILES src: N # C c 1 c c s c 1 N c 1 c c ( F ) c ( F ) c c 1 [N+] ( = O ) [O-]
SAFE src  : c 1 3 c c ( F ) c ( F ) c c 1 [N+] ( = O ) [O-] . N # C c 1 c c s c 1 2 . N 2 3
GT precursors: C1CCOC1.N#Cc1ccsc1N.O=[N+]([O-])c1cc(F)c(F)cc1F.[H-].[Na+]
GT safe col  : C1CCOC1~N#Cc1ccsc1N~O=[N+]([O-])c1cc(F)c(F)cc1F~[H-]~[Na+]>>c13cc(F)c(F)cc1[N+](=O)[O-].N#Cc1ccsc12.N23


In [40]:
for name in MODELS:
    if os.path.exists(CACHE_FILES[name]):
        os.remove(CACHE_FILES[name])
        print(f"Deleted: {CACHE_FILES[name]}")

Deleted: /data/ryanschen/safe-retro/smiles_run/eval_cache.pkl
Deleted: /data/ryanschen/safe-retro/safe_aug_run/eval_cache.pkl
